# E07 01 - LangChain puro (Resolution)

En el E07 original aprendemos la anatomia de un grafo. Antes de llegar al grafo,
vamos a resolver la misma idea como una **cadena lineal de LangChain**.

Flujo:

```txt
input del alumno
  -> prompt
  -> modelo
  -> parser
  -> funcion de formateo
  -> output final
```

La diferencia importante:

- en LangChain puro, una pieza llama a la siguiente en linea;
- en LangGraph, esas piezas pasan a ser nodos conectados por edges.


## Antes de tocar codigo: que estamos construyendo

Este notebook esta pensado para que puedas entenderlo aunque lo abras sin ver la clase.

Tema: **LangChain como paso previo a LangGraph**.

La regla didactica es:

1. primero explicamos el concepto;
2. despues mostramos el codigo minimo;
3. despues conectamos ese codigo con el paso anterior;
4. finalmente ejecutamos y leemos el resultado.

Cuando veas una funcion, preguntate:

- que recibe;
- que devuelve;
- que parte del flujo representa;
- si es logica de negocio, orquestacion o instrumentacion.


## Paso 1 - Instalacion

Usamos:

| Libreria | Por que aparece |
|---|---|
| `langchain` | Define prompts, parsers, runnables y LCEL |
| `langchain-openai` | Permite usar `ChatOpenAI` como modelo |

Todavia no usamos LangGraph ni Langfuse. Queremos entender primero el flujo lineal.


In [ ]:
# Esta celda instala las dependencias del notebook.
# En Google Colab cada notebook arranca con un entorno limpio, por eso instalamos al inicio.
# En local, si ya instalaste estos paquetes, pip simplemente confirmara que existen.
!pip install -q langchain langchain-openai

print('Dependencias instaladas: langchain langchain-openai')


## Paso 2 - Credencial de OpenAI

El LLM es una API externa. Para llamarlo necesitamos una API key.
No la guardamos en el notebook porque seria una mala practica de seguridad.


In [ ]:
import os
from getpass import getpass

# Nunca escribimos una API key real dentro del notebook.
# getpass permite pegar la key en ejecucion sin que quede visible en la salida.
# os.environ guarda la key solo para esta sesion de Python.
os.environ['OPENAI_API_KEY'] = getpass('OpenAI API Key: ')

print('OpenAI configurado para esta sesion.')


## Paso 3 - Imports, explicados uno por uno

Cada import representa una pieza conceptual:

- `ChatPromptTemplate`: transforma variables Python en mensajes para el LLM.
- `ChatOpenAI`: wrapper del modelo.
- `StrOutputParser`: convierte la respuesta del modelo a texto.
- `RunnableLambda`: permite insertar una funcion Python dentro de una chain.


In [ ]:
# ChatPromptTemplate nos deja crear prompts con variables como {name}.
from langchain_core.prompts import ChatPromptTemplate

# StrOutputParser extrae texto plano desde el AIMessage que devuelve el LLM.
from langchain_core.output_parsers import StrOutputParser

# RunnableLambda adapta una funcion Python para que pueda entrar en una chain LCEL.
from langchain_core.runnables import RunnableLambda

# ChatOpenAI es la integracion de LangChain con modelos de OpenAI.
from langchain_openai import ChatOpenAI

# temperature=0 hace que el modelo sea mas estable para ejercicios de clase.
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

print('Imports y modelo listos.')


## Paso 4 - Prompt, parser y formateo

Vamos a separar responsabilidades:

- el prompt decide que le pedimos al LLM;
- el LLM genera contenido;
- el parser limpia la salida;
- la funcion de formateo aplica una transformacion final.

Esta separacion es la antesala de los nodos de LangGraph.


In [ ]:
# El prompt tiene dos mensajes:
# - system: comportamiento general del asistente;
# - human: pedido concreto con una variable.
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Sos un asistente breve, claro y amable.'),
    ('human', 'Saluda a {name} en una sola oracion.'),
])

# El parser transforma AIMessage(content='...') en un string normal.
parser = StrOutputParser()

def format_output(text: str) -> str:
    # Esta funcion representa un segundo paso de procesamiento.
    # En LangGraph este paso podria convertirse en un nodo separado.
    return f'>> {text} <<'

# RunnableLambda permite que format_output participe en la chain.
format_step = RunnableLambda(format_output)

# LCEL conecta todas las piezas.
chain = prompt | llm | parser | format_step

# invoke ejecuta toda la cadena.
result = chain.invoke({'name': 'Ada'})
print(result)
